## Bitcoin Forensics: 
**Detecting Ransomware Addresses with a Perceptron Network**

### 1. Problem to Solve

Bitcoin transactions are public, but identifying addresses connected to ransomware activity is still difficult. This project uses the BitcoinHeist Ransomware Dataset to classify Bitcoin addresses as either normal or ransomware-related.

The goal is to train a perceptron-based model that detects suspicious addresses from transaction graph features such as chain length, number of neighbors, transaction count, looped transactions, weight, and income.

### 2. Setup
Imports all the necessary dependencies for the project

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

print("Environment check")
print("-----------------")
print(f"Python version: {sys.version.split()[0]}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

data_path = project_root / "data" / "BitcoinHeistData.csv"
figures_path = project_root / "outputs" / "figures"

print(f"Project root: {project_root}")
print(f"Data path exists: {data_path.exists()}")
print(f"Figures path exists: {figures_path.exists()}")

Environment check
-----------------
Python version: 3.13.5
NumPy version: 2.5.2
Pandas version: 3.0.5
Project root: /Users/victorvulturescu/Documents/RustWorks/IpWorkshop/bitcoin-forensics
Data path exists: True
Figures path exists: True


## 3. Load the dataset

This section loads the BitcoinHeist dataset into a pandas DataFrame so it can be inspected, cleaned, and prepared for machine learning.

In [5]:
data_path = project_root / "data" / "BitcoinHeistData.csv"

USE_SAMPLE = False
SAMPLE_ROWS = 200_000

if USE_SAMPLE:
    df = pd.read_csv(data_path, nrows=SAMPLE_ROWS)
else:
    df = pd.read_csv(data_path)

df = pd.read_csv(data_path)

print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape[0]:,} rows and {df.shape[1]} columns")

df.head()

Dataset loaded successfully.
Shape: 2,916,697 rows and 10 columns


,address,year,day,length,weight,count,looped,neighbors,income,label
0,111K8kZAEnJg245r2cM6y9zgJGHZtJPy6,2017,11,18,0.008333,1,0,2,100050000.0,princetonCerber
1,1123pJv8jzeFQaCV4w644pzQJzVWay2zcA,2016,132,44,0.000244,1,0,1,100000000.0,princetonLocky
2,112536im7hy6wtKbpH1qYDWtTyMRAcA2p7,2016,246,0,1.000000,1,0,2,200000000.0,princetonCerber
3,1126eDRw2wqSkWosjTCre8cjjQW8sSeWH7,2016,322,72,0.003906,1,0,2,71200000.0,princetonCerber
4,1129TSjKtx65E35GiUo4AYVeyo48twbrGX,2016,238,144,0.072848,456,0,1,200000000.0,princetonLocky


## 4.Understanding the dataset
This section explores the structure of the dataset, the meaning of each column, and the distribution of normal versus ransomware-related Bitcoin addresses.
The meaning of each of the collume is broken down as followes

| Column | Meaning | How we use it |
|---|---|---|
| `address` | Bitcoin address identifier | Removed before training, as its an ID, not a behaviour feature |
|`year`|Year of the transaction|Feature, may reflect time trends|
|`day`|Day of the year in which the transation was made|Feature, may reflect time trends|
|`length`|Length of the transaction chain connected to the address|Feature|
|`weight`|A graph-based weight measuring transaction flow/splitting behavio|Feature|
|`count`|Number of transactions or reachable outputs in the chain|Feature|
|`looped`|Amount/count of transactions that loop back in the graph|Feature|
|`neighbors`|Number of neighboring addresses connected in the graph|Feature|
|`income`|Amount received by the address, in satoshis|Feature|
|`label`|`white` for clean transactions, ransomware family name otherwise|Target Variable|

In [6]:
print(f"Rows: {df.shape[0]:,}")
print(f"Collumns: {df.shape[1]}")

df.head()

Rows: 2,916,697
Collumns: 10


,address,year,day,length,weight,count,looped,neighbors,income,label
0,111K8kZAEnJg245r2cM6y9zgJGHZtJPy6,2017,11,18,0.008333,1,0,2,100050000.0,princetonCerber
1,1123pJv8jzeFQaCV4w644pzQJzVWay2zcA,2016,132,44,0.000244,1,0,1,100000000.0,princetonLocky
2,112536im7hy6wtKbpH1qYDWtTyMRAcA2p7,2016,246,0,1.000000,1,0,2,200000000.0,princetonCerber
3,1126eDRw2wqSkWosjTCre8cjjQW8sSeWH7,2016,322,72,0.003906,1,0,2,71200000.0,princetonCerber
4,1129TSjKtx65E35GiUo4AYVeyo48twbrGX,2016,238,144,0.072848,456,0,1,200000000.0,princetonLocky


## 5. Preparing the Data
This section converts the dataset in to a binary classification problem, removes problems that are not necesarily helpfull for the problem that we want to solve and separates the input feratures from the target

In [7]:

# First we will create a coppy so the original dataframe stays intact
df_prepared = df.copy()

# Than we will convert the original labels in to binary form
# For the lable column, we don't really care about the particula ransomware family so the white lables become 1 while the ransomware ones become 0
df_prepared["is_ransomware"] = (df_prepared["label"] != "white").astype(int)

# For the address is an identifier, not a feature column so we can safely drop it
feature_columns = [
    "year",
    "day",
    "length",
    "weight",
    "count",
    "looped",
    "neighbors",
    "income",
]

# The features
X = df_prepared[feature_columns]
# The target
y = df_prepared["is_ransomware"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

df_prepared[["label", "is_ransomware"]].head()

Features shape: (2916697, 8)
Target shape: (2916697,)


,label,is_ransomware
0,princetonCerber,1
1,princetonLocky,1
2,princetonCerber,1
3,princetonCerber,1
4,princetonLocky,1


Blah blah

In [8]:
class_counts = df_prepared["is_ransomware"].value_counts().sort_index()
class_percentages = df_prepared["is_ransomware"].value_counts(normalize=True).sort_index() * 100

class_summary = pd.DataFrame({
    "class": ["normal", "ransomware"],
    "count": class_counts.values,
    "percentage": class_percentages.round(2).values,
})

class_summary

,class,count,percentage
0,normal,2875284,98.58
1,ransomware,41413,1.42
